# AIHub 186 Welfare Callcenter Whisper CER/WER Analysis

This notebook follows the same comparison style as `02-asr_whisper_baseline_zeroth-compare.ipynb`, but targets the AIHub 186 welfare callcenter dataset.

Goals:

1. Build a manifest from JSON/WAV pairs.
2. Compute Whisper baseline CER/WER by category level: facility, consultation type, and topic.
3. Rank finetuning candidates using baseline error and available data volume.
4. After finetuning, add the checkpoint path to `MODEL_IDS` and compare improvement against baseline.
5. Save recommended category filters and TSV files for the next finetuning experiment.

Main server paths:

- AIHub root: `/home/data/data/aihub`
- Existing prepared TSV: `/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv`
- Output directory: `/home/data/expr/week2/02-1-welfare_asr_error_analysis`


## 1. Imports


In [ ]:
import os
import re
import json
import time
import math
import hashlib
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display
from jiwer import wer, cer
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq


## 2. Configuration

For a first run, keep `MAX_EVAL_TOTAL` small enough to verify the pipeline. Increase it later for a more stable category-level estimate.

- `CATEGORY_FILTERS`: empty means stratified sampling across all categories.
- `MODEL_IDS`: the first model is treated as baseline. Add finetuned checkpoint paths after it.
- `MAX_EVAL_PER_TOPIC`: upper bound for each topic-level evaluation sample.


In [ ]:
BASE_DIR = Path('/home/data')
AIHUB_ROOT = Path('/home/data/data/aihub')
ASR_TSV = Path('/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv')
OUT_DIR = Path(os.environ.get('OUT_DIR', '/home/data/expr/week2/02-1-welfare_asr_error_analysis'))
OUT_DIR.mkdir(parents=True, exist_ok=True)


def discover_aihub186_data_dir() -> Path:
    env_dir = os.environ.get('AIHUB186_DATA_DIR')
    if env_dir:
        return Path(env_dir)
    candidates: list[Path] = []
    for dataset_dir in sorted(AIHUB_ROOT.glob('186.*')):
        candidates.extend(sorted(dataset_dir.glob('01.*')))
        candidates.append(dataset_dir)
    for candidate in candidates:
        if candidate.exists() and any(candidate.rglob('*.json')):
            return candidate
    raise FileNotFoundError(
        'Could not discover AIHub 186 dataset under /home/data/data/aihub. '
        'Set AIHUB186_DATA_DIR to the dataset or 01.* data directory.'
    )

AIHUB_DATA_DIR = discover_aihub186_data_dir()

MODEL_IDS = [
    'openai/whisper-large-v3-turbo',
    # After finetuning, add a checkpoint path, for example:
    # '/home/data/expr/week2/06-finetune_whisper/checkpoint-1000',
]

LANGUAGE = 'korean'
TASK = 'transcribe'
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '8'))
SEED = int(os.environ.get('SEED', '42'))

# 0 means no total cap. Recommended first-run values: 100 to 500.
MAX_EVAL_TOTAL = int(os.environ.get('MAX_EVAL_TOTAL', '300'))
MAX_EVAL_PER_TOPIC = int(os.environ.get('MAX_EVAL_PER_TOPIC', '15'))
MIN_DURATION_SEC = float(os.environ.get('MIN_DURATION_SEC', '0.1'))
MAX_DURATION_SEC = float(os.environ.get('MAX_DURATION_SEC', '30.0'))

# One dict is AND conditions; multiple dicts are OR conditions.
# Example: [{'category1': '...', 'category2': '...'}]
CATEGORY_FILTERS: list[dict[str, str]] = []

print('AIHUB_DATA_DIR:', AIHUB_DATA_DIR)
print('ASR_TSV:', ASR_TSV)
print('OUT_DIR:', OUT_DIR)
print('MODEL_IDS:', MODEL_IDS)
print('MAX_EVAL_TOTAL:', MAX_EVAL_TOTAL)
print('MAX_EVAL_PER_TOPIC:', MAX_EVAL_PER_TOPIC)


## 3. Root Discovery And Manifest Build

The expected hierarchy is interpreted as `facility / consultation type / topic / bundle / json-wav pair`. JSON `audioPath` can be a Windows source path, so this notebook uses the WAV file with the same stem next to each JSON file.


In [ ]:
def find_training_root(data_dir: Path) -> Path:
    env_root = os.environ.get('WELFARE_ROOT')
    candidates: list[Path] = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        data_dir / '1.Training',
        data_dir / 'Training',
    ])
    candidates.extend(sorted(data_dir.glob('*/1.Training')))
    candidates.extend(sorted(data_dir.glob('*/Training')))
    candidates.append(data_dir)
    for candidate in candidates:
        if candidate.exists() and any(candidate.rglob('*.json')):
            return candidate
    raise FileNotFoundError(
        'Could not find JSON files. Check WELFARE_ROOT or AIHUB_DATA_DIR: '
        f'{data_dir}'
    )

ROOT = find_training_root(AIHUB_DATA_DIR)
print('ROOT:', ROOT)


In [ ]:
def remove_number_prefix(name: str) -> str:
    return name.split('.', 1)[1].strip() if '.' in name else name.strip()


def extract_number_prefix(name: str) -> int | None:
    prefix = name.split('.', 1)[0]
    try:
        return int(prefix)
    except ValueError:
        return None


def first_dict(value: Any) -> dict[str, Any]:
    if isinstance(value, list) and value and isinstance(value[0], dict):
        return value[0]
    if isinstance(value, dict):
        return value
    return {}


def safe_float(value: Any) -> float | None:
    try:
        if value is None or value == '':
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def read_json(path: Path) -> dict[str, Any] | None:
    for encoding in ('utf-8-sig', 'utf-8', 'cp949'):
        try:
            with path.open('r', encoding=encoding) as f:
                return json.load(f)
        except UnicodeDecodeError:
            continue
        except json.JSONDecodeError as e:
            print('[JSON decode error]', path, e)
            return None
    print('[JSON encoding error]', path)
    return None


In [ ]:
records: list[dict[str, Any]] = []
missing_wav: list[str] = []
missing_json: list[str] = []
metadata_mismatches: list[dict[str, str]] = []

json_paths = sorted(ROOT.rglob('*.json'))
print('JSON files:', len(json_paths))

for json_path in json_paths:
    rel = json_path.relative_to(ROOT).parts
    if len(rel) < 5:
        continue

    facility_dir, consultation_dir, topic_dir, bundle_id = rel[0], rel[1], rel[2], rel[3]
    wav_path = json_path.with_suffix('.wav')
    if not wav_path.exists():
        missing_wav.append(str(json_path))

    data = read_json(json_path)
    if data is None:
        continue

    input_text = first_dict(data.get('inputText'))
    dialog = first_dict(data.get('dialogs'))
    info = first_dict(data.get('info'))
    metadata = info.get('metadata', {}) if isinstance(info.get('metadata', {}), dict) else {}

    path_category1 = remove_number_prefix(facility_dir)
    path_category2 = remove_number_prefix(consultation_dir)
    path_category3 = remove_number_prefix(topic_dir)

    # Prefer JSON metadata when present; otherwise use directory-derived categories.
    category1 = str(metadata.get('category1') or path_category1).strip()
    category2 = str(metadata.get('category2') or path_category2).strip()
    category3 = str(metadata.get('category3') or path_category3).strip()

    for key, path_value, meta_value in [
        ('category1', path_category1, category1),
        ('category2', path_category2, category2),
        ('category3', path_category3, category3),
    ]:
        if meta_value and path_value and meta_value != path_value:
            metadata_mismatches.append({
                'json_path': str(json_path),
                'field': key,
                'directory_value': path_value,
                'metadata_value': meta_value,
            })

    records.append({
        'file_id': json_path.stem,
        'sample_id': json_path.stem,
        'facility_directory': facility_dir,
        'consultation_directory': consultation_dir,
        'topic_directory': topic_dir,
        'category1': category1,
        'category2': category2,
        'category3': category3,
        'facility_code': json_path.stem[:3],
        'consultation_type_code': extract_number_prefix(consultation_dir),
        'consultation_topic_code': extract_number_prefix(topic_dir),
        'bundle_id': bundle_id,
        'json_path': str(json_path),
        'audio_path': str(wav_path) if wav_path.exists() else None,
        'transcript': str(input_text.get('orgtext') or '').strip(),
        'json_audio_path': dialog.get('audioPath'),
        'speaker_type': metadata.get('speaker_type'),
        'speaker_id': metadata.get('speaker_id'),
        'speaker_age': metadata.get('speaker_age'),
        'speaker_sex': metadata.get('speaker_sex'),
        'duration_sec': safe_float(metadata.get('sptime_all')),
        'speech_start': safe_float(metadata.get('sptime_start')),
        'speech_end': safe_float(metadata.get('sptime_end')),
        'recording_device': metadata.get('rec_device'),
        'recording_place': metadata.get('rec_place'),
    })

for wav_path in sorted(ROOT.rglob('*.wav')):
    if not wav_path.with_suffix('.json').exists():
        missing_json.append(str(wav_path))

manifest_df = pd.DataFrame(records)
manifest_path = OUT_DIR / 'welfare_callcenter_manifest.csv'
manifest_df.to_csv(manifest_path, index=False, encoding='utf-8-sig')

print('manifest rows:', len(manifest_df))
print('missing wav:', len(missing_wav))
print('missing json:', len(missing_json))
print('metadata mismatches:', len(metadata_mismatches))
print('saved:', manifest_path)
manifest_df.head()


## 4. Data Distribution

Check sample counts and hours by facility, consultation type, and topic. Candidate selection should consider both baseline error and data volume.


In [ ]:
def summarize_distribution(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    x = df.copy()
    x['duration_sec'] = pd.to_numeric(x['duration_sec'], errors='coerce')
    summary = (
        x.groupby(group_cols, dropna=False)
        .agg(
            n_samples=('file_id', 'count'),
            n_speakers=('speaker_id', 'nunique'),
            duration_hours=('duration_sec', lambda s: float(s.fillna(0).sum() / 3600)),
            avg_duration_sec=('duration_sec', 'mean'),
            transcript_chars=('transcript', lambda s: int(s.fillna('').astype(str).str.len().sum())),
        )
        .reset_index()
        .sort_values(['duration_hours', 'n_samples'], ascending=False)
    )
    return summary

valid_manifest_df = manifest_df[
    manifest_df['audio_path'].notna()
    & manifest_df['transcript'].fillna('').astype(str).str.strip().ne('')
].copy()
valid_manifest_df['duration_sec'] = pd.to_numeric(valid_manifest_df['duration_sec'], errors='coerce')

facility_dist = summarize_distribution(valid_manifest_df, ['category1'])
type_dist = summarize_distribution(valid_manifest_df, ['category1', 'category2'])
topic_dist = summarize_distribution(valid_manifest_df, ['category1', 'category2', 'category3'])

facility_dist.to_csv(OUT_DIR / 'distribution_facility.csv', index=False, encoding='utf-8-sig')
type_dist.to_csv(OUT_DIR / 'distribution_type.csv', index=False, encoding='utf-8-sig')
topic_dist.to_csv(OUT_DIR / 'distribution_topic.csv', index=False, encoding='utf-8-sig')

display(facility_dist)
display(type_dist.head(20))
display(topic_dist.head(30))


## 5. Evaluation Sample

The evaluation set is stratified by topic using `MAX_EVAL_PER_TOPIC`. Evaluation file IDs are later excluded from the recommended training TSV when `EXCLUDE_EVAL_FROM_TRAIN=True`.


In [ ]:
def apply_category_filters(df: pd.DataFrame, filters: list[dict[str, str]]) -> pd.DataFrame:
    if not filters:
        return df.copy()
    total_mask = pd.Series(False, index=df.index)
    for condition in filters:
        condition_mask = pd.Series(True, index=df.index)
        for col, expected in condition.items():
            if col not in df.columns:
                raise ValueError(f'Unknown filter column: {col}')
            condition_mask &= df[col].fillna('').astype(str).str.strip().eq(str(expected).strip())
        total_mask |= condition_mask
    return df[total_mask].copy()


def stable_sample(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if n <= 0 or len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed)


eval_pool = valid_manifest_df.copy()
eval_pool['duration_sec'] = pd.to_numeric(eval_pool['duration_sec'], errors='coerce')
eval_pool = eval_pool[
    eval_pool['duration_sec'].between(MIN_DURATION_SEC, MAX_DURATION_SEC, inclusive='both')
    & eval_pool['audio_path'].map(lambda p: Path(str(p)).is_file())
].copy()
eval_pool = apply_category_filters(eval_pool, CATEGORY_FILTERS)

eval_parts = []
for _, group in eval_pool.groupby(['category1', 'category2', 'category3'], dropna=False):
    eval_parts.append(stable_sample(group, MAX_EVAL_PER_TOPIC, SEED))

eval_df = pd.concat(eval_parts, ignore_index=True) if eval_parts else eval_pool.iloc[0:0].copy()
if MAX_EVAL_TOTAL > 0 and len(eval_df) > MAX_EVAL_TOTAL:
    eval_df = stable_sample(eval_df, MAX_EVAL_TOTAL, SEED)

eval_df = eval_df.sort_values(['category1', 'category2', 'category3', 'file_id']).reset_index(drop=True)
eval_df['ref'] = eval_df['transcript'].astype(str)

eval_manifest_path = OUT_DIR / 'eval_manifest.csv'
eval_df.to_csv(eval_manifest_path, index=False, encoding='utf-8-sig')

print('eval rows:', len(eval_df), '/', len(eval_pool))
print('eval hours:', round(eval_df['duration_sec'].fillna(0).sum() / 3600, 3))
print('saved:', eval_manifest_path)
display(summarize_distribution(eval_df, ['category1', 'category2', 'category3']).head(50))


## 6. Metric Normalization

For Korean ASR, spacing errors can dominate character metrics. This notebook saves both `cer_raw` and `cer_no_space`; use `cer_no_space` as the main CER target and keep WER for word/spacing behavior.


In [ ]:
_punct_re = re.compile(r'[^\w\s]', flags=re.UNICODE)
_space_re = re.compile(r'\s+')


def normalize_for_metric(text: Any, remove_space: bool = False) -> str:
    text = '' if pd.isna(text) else str(text)
    text = text.lower().strip()
    text = _punct_re.sub(' ', text)
    text = _space_re.sub(' ', text).strip()
    if remove_space:
        text = text.replace(' ', '')
    return text


def safe_wer(ref: Any, hyp: Any) -> float:
    r = normalize_for_metric(ref, remove_space=False)
    h = normalize_for_metric(hyp, remove_space=False)
    if not r and not h:
        return 0.0
    return float(wer([r], [h]))


def safe_cer(ref: Any, hyp: Any, remove_space: bool = False) -> float:
    r = normalize_for_metric(ref, remove_space=remove_space)
    h = normalize_for_metric(hyp, remove_space=remove_space)
    if not r and not h:
        return 0.0
    return float(cer([r], [h]))


def aggregate_metrics(refs: pd.Series, hyps: pd.Series) -> dict[str, float]:
    refs_norm = [normalize_for_metric(x, remove_space=False) for x in refs]
    hyps_norm = [normalize_for_metric(x, remove_space=False) for x in hyps]
    refs_no_space = [normalize_for_metric(x, remove_space=True) for x in refs]
    hyps_no_space = [normalize_for_metric(x, remove_space=True) for x in hyps]
    return {
        'wer_norm': float(wer(refs_norm, hyps_norm)),
        'cer_raw': float(cer(refs_norm, hyps_norm)),
        'cer_no_space': float(cer(refs_no_space, hyps_no_space)),
    }


## 7. Whisper Transcription

Hypothesis and evaluation CSV files are cached per model. Delete cached files or set `FORCE_TRANSCRIBE=True` to rerun transcription.


In [ ]:
FORCE_TRANSCRIBE = False

def model_tag(model_id: str) -> str:
    return model_id.strip('/').replace('/', '_').replace('\\', '_').replace(':', '_')


def transcribe_one_model(model_id: str, files: list[str]) -> tuple[pd.DataFrame, float]:
    tag = model_tag(model_id)
    hyp_path = OUT_DIR / f'hyp_{tag}.csv'
    if hyp_path.exists() and not FORCE_TRANSCRIBE:
        df_hyp = pd.read_csv(hyp_path, encoding='utf-8-sig')
        print(f'[{model_id}] cached:', hyp_path, 'rows=', len(df_hyp))
        return df_hyp, 0.0

    if torch.cuda.is_available():
        device = 0
        dtype = torch.float16
        visible = os.environ.get('CUDA_VISIBLE_DEVICES', '(not set)')
    else:
        device = -1
        dtype = torch.float32
        visible = '(cpu)'

    print(f'MODEL: {model_id} | CUDA_VISIBLE_DEVICES: {visible} | device: {device} | dtype: {dtype}')
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, dtype=dtype)
    if torch.cuda.is_available():
        model = model.to('cuda')

    asr = hf_pipeline(
        'automatic-speech-recognition',
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
    )

    rows = []
    t0 = time.time()
    for start in range(0, len(files), BATCH_SIZE):
        chunk = files[start:start + BATCH_SIZE]
        outs = asr(
            chunk,
            batch_size=BATCH_SIZE,
            generate_kwargs={'language': LANGUAGE, 'task': TASK, 'temperature': 0.0},
        )
        for path, out in zip(chunk, outs):
            rows.append({
                'file_id': Path(path).stem,
                'audio_path': path,
                'hyp': str(out.get('text', '')).strip(),
                'model_id': model_id,
            })
        print(f'[{model_id}] {min(start + BATCH_SIZE, len(files))}/{len(files)} done')

    elapsed = time.time() - t0
    df_hyp = pd.DataFrame(rows)
    df_hyp.to_csv(hyp_path, index=False, encoding='utf-8-sig')
    print('saved:', hyp_path, 'elapsed_sec=', round(elapsed, 2))
    return df_hyp, elapsed


In [ ]:
assert len(eval_df) > 0, 'No evaluation rows. Check CATEGORY_FILTERS and data paths.'

df_eval_by_model: dict[str, pd.DataFrame] = {}
all_results = []
files = eval_df['audio_path'].astype(str).tolist()

for model_id in MODEL_IDS:
    print('\n' + '=' * 90)
    df_hyp, elapsed = transcribe_one_model(model_id, files)
    df_eval = eval_df.merge(
        df_hyp[['file_id', 'hyp', 'model_id']],
        on='file_id',
        how='inner',
    )
    if len(df_eval) != len(eval_df):
        print('matched rows:', len(df_eval), '/', len(eval_df))

    df_eval['wer_norm_sample'] = [safe_wer(r, h) for r, h in zip(df_eval['ref'], df_eval['hyp'])]
    df_eval['cer_raw_sample'] = [safe_cer(r, h, remove_space=False) for r, h in zip(df_eval['ref'], df_eval['hyp'])]
    df_eval['cer_no_space_sample'] = [safe_cer(r, h, remove_space=True) for r, h in zip(df_eval['ref'], df_eval['hyp'])]

    metrics = aggregate_metrics(df_eval['ref'], df_eval['hyp'])
    row = {
        'model_id': model_id,
        'model_tag': model_tag(model_id),
        'n_files': len(df_eval),
        'elapsed_sec': round(elapsed, 3),
        'sec_per_file': round(elapsed / max(len(df_eval), 1), 4) if elapsed else 0.0,
        **metrics,
        'mean_wer_norm': float(df_eval['wer_norm_sample'].mean()),
        'mean_cer_raw': float(df_eval['cer_raw_sample'].mean()),
        'mean_cer_no_space': float(df_eval['cer_no_space_sample'].mean()),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    all_results.append(row)
    df_eval_by_model[model_id] = df_eval

    eval_path = OUT_DIR / f'eval_{model_tag(model_id)}.csv'
    df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
    print(row)
    print('saved:', eval_path)

result_summary_df = pd.DataFrame(all_results).sort_values('cer_no_space')
result_summary_path = OUT_DIR / 'model_overall_summary.csv'
result_summary_df.to_csv(result_summary_path, index=False, encoding='utf-8-sig')
display(result_summary_df)
print('saved:', result_summary_path)


## 8. Category-Level CER/WER

Aggregate CER/WER and mean per-sample CER/WER are saved for facility, consultation type, and topic levels. For finetuning choice, inspect `cer_no_space`, `wer_norm`, `n_files`, and `duration_hours` together.


In [ ]:
def grouped_error_summary(df_eval: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for key, group in df_eval.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        metrics = aggregate_metrics(group['ref'], group['hyp'])
        row = {col: value for col, value in zip(group_cols, key)}
        row.update({
            'model_id': group['model_id'].iloc[0],
            'model_tag': model_tag(group['model_id'].iloc[0]),
            'n_files': len(group),
            'duration_hours': float(pd.to_numeric(group['duration_sec'], errors='coerce').fillna(0).sum() / 3600),
            'mean_wer_norm': float(group['wer_norm_sample'].mean()),
            'mean_cer_raw': float(group['cer_raw_sample'].mean()),
            'mean_cer_no_space': float(group['cer_no_space_sample'].mean()),
            **metrics,
        })
        rows.append(row)
    return pd.DataFrame(rows)

level_defs = {
    'facility': ['category1'],
    'type': ['category1', 'category2'],
    'topic': ['category1', 'category2', 'category3'],
}

level_summaries = {}
for level_name, group_cols in level_defs.items():
    parts = []
    for model_id, df_eval in df_eval_by_model.items():
        parts.append(grouped_error_summary(df_eval, group_cols))
    summary = pd.concat(parts, ignore_index=True).sort_values(['cer_no_space', 'wer_norm'], ascending=False)
    level_summaries[level_name] = summary
    path = OUT_DIR / f'error_summary_by_{level_name}.csv'
    summary.to_csv(path, index=False, encoding='utf-8-sig')
    print(level_name, 'saved:', path)
    display(summary.head(20))


## 9. Baseline Improvement After Finetuning

After adding a finetuned checkpoint to `MODEL_IDS`, this section computes `(baseline - candidate) / baseline`. The intermediate target is at least 10% improvement for both CER and WER.


In [ ]:
baseline_model = MODEL_IDS[0]
baseline_tag = model_tag(baseline_model)

comparison_tables = {}
if len(MODEL_IDS) >= 2:
    for level_name, summary in level_summaries.items():
        id_cols = level_defs[level_name]
        base = summary[summary['model_id'].eq(baseline_model)].copy()
        base = base[id_cols + ['n_files', 'duration_hours', 'wer_norm', 'cer_no_space']].rename(columns={
            'wer_norm': 'baseline_wer_norm',
            'cer_no_space': 'baseline_cer_no_space',
            'n_files': 'baseline_n_files',
            'duration_hours': 'baseline_duration_hours',
        })
        rows = []
        for model_id in MODEL_IDS[1:]:
            cand = summary[summary['model_id'].eq(model_id)].copy()
            cand = cand[id_cols + ['wer_norm', 'cer_no_space']].rename(columns={
                'wer_norm': 'candidate_wer_norm',
                'cer_no_space': 'candidate_cer_no_space',
            })
            merged = base.merge(cand, on=id_cols, how='inner')
            merged['candidate_model_id'] = model_id
            merged['wer_improvement_ratio'] = (
                (merged['baseline_wer_norm'] - merged['candidate_wer_norm'])
                / merged['baseline_wer_norm'].replace(0, np.nan)
            )
            merged['cer_improvement_ratio'] = (
                (merged['baseline_cer_no_space'] - merged['candidate_cer_no_space'])
                / merged['baseline_cer_no_space'].replace(0, np.nan)
            )
            merged['pass_10pct_both'] = (
                merged['wer_improvement_ratio'].ge(0.10)
                & merged['cer_improvement_ratio'].ge(0.10)
            )
            rows.append(merged)
        comp = pd.concat(rows, ignore_index=True).sort_values(
            ['pass_10pct_both', 'cer_improvement_ratio', 'wer_improvement_ratio'],
            ascending=[False, False, False],
        )
        comparison_tables[level_name] = comp
        path = OUT_DIR / f'baseline_improvement_by_{level_name}.csv'
        comp.to_csv(path, index=False, encoding='utf-8-sig')
        print(level_name, 'saved:', path)
        display(comp.head(30))
else:
    print('No finetuned model to compare. Add a checkpoint path to MODEL_IDS after finetuning.')


## 10. Finetuning Candidate Ranking

Before a finetuned model exists, useful candidates are groups where baseline error is high and enough training data exists.

The ranking combines:

- baseline `cer_no_space`
- baseline `wer_norm`
- total hours available in the full manifest
- estimated character error burden in the evaluation sample

Because the final service is an ASR plus LLM interaction system, prioritize user-like, diverse, and high-error call types over volume alone.


In [ ]:
def minmax(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce').fillna(0)
    lo, hi = float(s.min()), float(s.max())
    if math.isclose(lo, hi):
        return pd.Series(0.0, index=series.index)
    return (s - lo) / (hi - lo)

baseline_topic_eval = level_summaries['topic'][
    level_summaries['topic']['model_id'].eq(baseline_model)
].copy()

all_topic_data = topic_dist.rename(columns={
    'n_samples': 'total_n_samples',
    'duration_hours': 'total_duration_hours',
    'transcript_chars': 'total_transcript_chars',
})

candidate_df = baseline_topic_eval.merge(
    all_topic_data[
        ['category1', 'category2', 'category3', 'total_n_samples', 'total_duration_hours', 'total_transcript_chars']
    ],
    on=['category1', 'category2', 'category3'],
    how='left',
)

eval_ref_chars_by_topic = (
    df_eval_by_model[baseline_model]
    .assign(ref_chars=lambda d: d['ref'].fillna('').astype(str).map(lambda x: len(normalize_for_metric(x, remove_space=True))))
    .groupby(['category1', 'category2', 'category3'], dropna=False)['ref_chars']
    .sum()
    .reset_index(name='eval_ref_chars')
)

candidate_df = candidate_df.merge(
    eval_ref_chars_by_topic,
    on=['category1', 'category2', 'category3'],
    how='left',
)
candidate_df['eval_ref_chars'] = candidate_df['eval_ref_chars'].fillna(0).astype(int)
candidate_df['estimated_char_error_burden'] = candidate_df['eval_ref_chars'] * candidate_df['cer_no_space']

candidate_df['score'] = (
    0.40 * minmax(candidate_df['cer_no_space'])
    + 0.30 * minmax(candidate_df['wer_norm'])
    + 0.20 * minmax(np.log1p(candidate_df['total_duration_hours'].fillna(0)))
    + 0.10 * minmax(candidate_df['estimated_char_error_burden'])
)

candidate_df['priority_reason'] = np.select(
    [
        candidate_df['total_duration_hours'].fillna(0).lt(0.5),
        candidate_df['cer_no_space'].ge(candidate_df['cer_no_space'].quantile(0.75)),
        candidate_df['wer_norm'].ge(candidate_df['wer_norm'].quantile(0.75)),
    ],
    [
        'Low data volume; inspect before prioritizing',
        'High CER group; strong character-level improvement potential',
        'High WER group; word/spacing improvement potential',
    ],
    default='Medium priority',
)

candidate_df = candidate_df.sort_values('score', ascending=False).reset_index(drop=True)
candidate_path = OUT_DIR / 'finetuning_candidate_topics.csv'
candidate_df.to_csv(candidate_path, index=False, encoding='utf-8-sig')
print('saved:', candidate_path)
display(candidate_df.head(20)[[
    'category1', 'category2', 'category3',
    'n_files', 'total_n_samples', 'total_duration_hours',
    'cer_no_space', 'wer_norm', 'score', 'priority_reason'
]])


## 11. Build Candidate Combination And Save TSV

Top topic groups are accumulated into a practical finetuning subset. Tune the target hour range based on available GPU time. Very small sets can overfit; very large sets slow down iteration.


In [ ]:
TARGET_TRAIN_HOURS_MIN = 2.0
TARGET_TRAIN_HOURS_MAX = 8.0
TOPIC_TOP_K = 8
EXCLUDE_EVAL_FROM_TRAIN = True

selected_topics = []
selected_hours = 0.0
for _, row in candidate_df.head(30).iterrows():
    if len(selected_topics) >= TOPIC_TOP_K:
        break
    if selected_hours >= TARGET_TRAIN_HOURS_MIN and selected_hours + float(row.get('total_duration_hours') or 0) > TARGET_TRAIN_HOURS_MAX:
        continue
    selected_topics.append({
        'category1': row['category1'],
        'category2': row['category2'],
        'category3': row['category3'],
    })
    selected_hours += float(row.get('total_duration_hours') or 0)
    if selected_hours >= TARGET_TRAIN_HOURS_MIN and len(selected_topics) >= 3:
        break

print('selected topics:', selected_topics)
print('selected total hours:', round(selected_hours, 3))

selected_df = apply_category_filters(valid_manifest_df, selected_topics)
selected_df['dataset_type'] = 'train'
selected_df = selected_df[
    selected_df['duration_sec'].between(MIN_DURATION_SEC, MAX_DURATION_SEC, inclusive='both')
].copy()

if EXCLUDE_EVAL_FROM_TRAIN:
    eval_ids = set(eval_df['file_id'].astype(str))
    selected_df = selected_df[~selected_df['file_id'].astype(str).isin(eval_ids)].copy()

finetune_tsv_df = selected_df[[
    'file_id', 'audio_path', 'transcript', 'duration_sec', 'dataset_type',
    'category1', 'category2', 'category3', 'speaker_type', 'speaker_id',
]].copy()

finetune_tsv_path = OUT_DIR / 'recommended_finetune_dataset.tsv'
finetune_tsv_df.to_csv(finetune_tsv_path, sep='\t', index=False, encoding='utf-8-sig')

filters_py_path = OUT_DIR / 'recommended_category_filters.py'
with filters_py_path.open('w', encoding='utf-8') as f:
    f.write('CATEGORY_FILTERS = ') 
    f.write(json.dumps(selected_topics, ensure_ascii=False, indent=4))
    f.write('\n')

print('finetune rows:', len(finetune_tsv_df))
print('finetune hours:', round(finetune_tsv_df['duration_sec'].fillna(0).sum() / 3600, 3))
print('saved:', finetune_tsv_path)
print('saved:', filters_py_path)
display(finetune_tsv_df.head())


## 12. Listen To High-Error Samples

Listen to high-CER baseline samples before finalizing finetuning data. Check whether errors come from domain mismatch, noisy audio, silence, or label problems.


In [ ]:
baseline_eval = df_eval_by_model[baseline_model].copy()
inspect_df = baseline_eval.sort_values('cer_no_space_sample', ascending=False).head(10)

for _, r in inspect_df.iterrows():
    print('=' * 100)
    print('TYPE:', r['category1'], '>', r['category2'], '>', r['category3'])
    print('FILE:', r['file_id'])
    print('CER(no-space):', round(float(r['cer_no_space_sample']), 4), 'WER:', round(float(r['wer_norm_sample']), 4))
    print('REF:', r['ref'])
    print('HYP:', r['hyp'])
    print('PATH:', r['audio_path'])
    display(Audio(filename=r['audio_path']))


## 13. Decision Checklist

1. Inspect top rows in `finetuning_candidate_topics.csv`.
2. Exclude or clean categories with many label/audio quality issues.
3. Use `recommended_category_filters.py` in the filtering notebook, or use `recommended_finetune_dataset.tsv` directly as the next training input.
4. After finetuning, add the checkpoint path to `MODEL_IDS` and rerun this notebook.
5. Check `baseline_improvement_by_topic.csv` and `baseline_improvement_by_type.csv` for `cer_improvement_ratio >= 0.10` and `wer_improvement_ratio >= 0.10`.
